# Mini-Lab E: Dataflow + RunInference

**Goal:** Build an Apache Beam pipeline with in-pipeline ML inference using `RunInference`, run it locally with `DirectRunner`, then deploy to Google Cloud Dataflow with `DataflowRunner`.

**Dataset:** Census income (recurring baseline)  
**Model:** sklearn GradientBoostingClassifier  
**Exam Relevance:** Very high — RunInference, Beam I/O, DirectRunner vs DataflowRunner, in-pipeline inference pattern

**Key exam pattern:** "You have a streaming/batch pipeline that needs ML predictions with minimal latency. Where should the model live?"  
→ **Load the model into Dataflow workers using RunInference. In-process inference adds ~1-5ms vs 30-100ms+ for external endpoint calls.**

---

## Parts
1. **Setup & Model Prep** — Install deps, train sklearn model, upload to GCS
2. **Beam Pipeline (DirectRunner)** — Build and test locally
3. **Deploy to Dataflow (DataflowRunner)** — Same code, cloud execution
4. **Compare & Analyse** — Verify predictions, document patterns
5. **Cleanup** — Remove all resources

---
## Part 1: Setup & Model Prep

### 1.1 Configuration

In [1]:
# ============================================================
# Configuration
# ============================================================
PROJECT_ID = "carty-470812"
REGION = "us-central1"
BUCKET = "carty-470812"
PREFIX = "mini-lab-e-dataflow"

# GCS paths
MODEL_PATH = f"gs://{BUCKET}/{PREFIX}/model/model.pkl"
TEST_DATA_PATH = f"gs://{BUCKET}/{PREFIX}/data/test.csv"
TRAIN_DATA_PATH = f"gs://{BUCKET}/{PREFIX}/data/train.csv"
LOCAL_OUTPUT_PATH = f"gs://{BUCKET}/{PREFIX}/output/local/"
DATAFLOW_OUTPUT_TABLE = f"{PROJECT_ID}.mini_lab_e.predictions"
TEMP_LOCATION = f"gs://{BUCKET}/{PREFIX}/temp"
STAGING_LOCATION = f"gs://{BUCKET}/{PREFIX}/staging"

print(f"Project:  {PROJECT_ID}")
print(f"Region:   {REGION}")
print(f"Bucket:   {BUCKET}")
print(f"Prefix:   {PREFIX}")

Project:  carty-470812
Region:   us-central1
Bucket:   carty-470812
Prefix:   mini-lab-e-dataflow


### 1.2 Install Dependencies

We need `apache-beam[gcp]` which includes the GCP I/O connectors (BigQuery, GCS) and the `DataflowRunner`. The `RunInference` API lives in `apache_beam.ml.inference`.

In [2]:
# Install apache-beam with GCP extras
# This includes google-cloud-bigquery, google-cloud-storage, etc.
!pip install "apache-beam[gcp]" "apache-beam[tfrecord]" "apache-beam[interactive]" --quiet

# Verify installation
import apache_beam
print(f"Apache Beam version: {apache_beam.__version__}")


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Apache Beam version: 2.71.0


### 1.3 Verify Dataflow API is Enabled

In [3]:
# Check that the Dataflow API is enabled
!gcloud services list --enabled --filter="name:dataflow.googleapis.com" --project={PROJECT_ID} --format="value(name)"

# If nothing prints above, enable it:
# !gcloud services enable dataflow.googleapis.com --project={PROJECT_ID}

projects/873708835509/services/dataflow.googleapis.com


### 1.4 Prepare Census Data

We'll query the census income dataset from BigQuery, do minimal preprocessing, and export train/test CSVs to GCS. The preprocessing needs to happen *before* model training because sklearn expects numeric input.

In [4]:
from google.cloud import bigquery
import pandas as pd
import numpy as np

client = bigquery.Client(project=PROJECT_ID)

# Query census data — same dataset as Labs 1-3
# We'll do the feature engineering in pandas to keep it simple
query = """
SELECT
    age,
    workclass,
    education_num,
    marital_status,
    occupation,
    relationship,
    race,
    sex,
    capital_gain,
    capital_loss,
    hours_per_week,
    native_country,
    income_bracket
FROM `bigquery-public-data.ml_datasets.census_adult_income`
WHERE age IS NOT NULL
  AND workclass IS NOT NULL
  AND education_num IS NOT NULL
  AND occupation IS NOT NULL
"""

df = client.query(query).to_dataframe()
print(f"Total rows: {len(df):,}")
print(f"\nLabel distribution:")
print(df['income_bracket'].value_counts())
print(f"\nSample:")
df.head()

Total rows: 32,561

Label distribution:
income_bracket
<=50K    24720
>50K      7841
Name: count, dtype: int64

Sample:


,age,workclass,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income_bracket
0,39,Private,5,Married-civ-spouse,Other-service,Wife,Black,Female,3411,0,34,United-States,<=50K
1,77,Private,5,Married-civ-spouse,Priv-house-serv,Wife,Black,Female,0,0,10,United-States,<=50K
2,38,Private,5,Married-civ-spouse,Other-service,Wife,Black,Female,0,0,24,Haiti,<=50K
3,28,Private,5,Married-civ-spouse,Protective-serv,Wife,Black,Female,0,0,40,United-States,<=50K
4,37,Private,5,Married-civ-spouse,Machine-op-inspct,Wife,Black,Female,0,0,48,United-States,<=50K


In [5]:
# Create binary label
df['label'] = (df['income_bracket'].str.strip() == '>50K').astype(int)
df = df.drop(columns=['income_bracket'])

# One-hot encode categorical features
# We need to track the column order so we can replicate it at inference time
categorical_cols = ['workclass', 'marital_status', 'occupation', 
                    'relationship', 'race', 'sex', 'native_country']
numeric_cols = ['age', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']

df_encoded = pd.get_dummies(df, columns=categorical_cols, dtype=int)

# Deterministic train/test split using hash (same approach as other labs)
# Hash the index to get reproducible splits
df_encoded['split_hash'] = df_encoded.index.map(lambda x: hash(str(x)) % 100)
train_df = df_encoded[df_encoded['split_hash'] < 80].drop(columns=['split_hash'])
test_df = df_encoded[df_encoded['split_hash'] >= 80].drop(columns=['split_hash'])

print(f"Train: {len(train_df):,} rows")
print(f"Test:  {len(test_df):,} rows")

# Save column order (excluding label) — we'll need this at inference time
feature_columns = [c for c in train_df.columns if c != 'label']
print(f"Features: {len(feature_columns)} columns")

Train: 26,105 rows
Test:  6,456 rows
Features: 91 columns


In [6]:
from google.cloud import storage

# Save train and test CSVs to GCS
# Include headers so we can parse them in the Beam pipeline
storage_client = storage.Client(project=PROJECT_ID)
bucket_obj = storage_client.bucket(BUCKET)

# Train CSV
train_csv = train_df.to_csv(index=False)
blob = bucket_obj.blob(f"{PREFIX}/data/train.csv")
blob.upload_from_string(train_csv, content_type='text/csv')
print(f"Uploaded train data to {TRAIN_DATA_PATH}")

# Test CSV (this is what we'll run inference on)
test_csv = test_df.to_csv(index=False)
blob = bucket_obj.blob(f"{PREFIX}/data/test.csv")
blob.upload_from_string(test_csv, content_type='text/csv')
print(f"Uploaded test data to {TEST_DATA_PATH}")

# Also save feature columns list for reference
import json
feature_json = json.dumps(feature_columns)
blob = bucket_obj.blob(f"{PREFIX}/data/feature_columns.json")
blob.upload_from_string(feature_json, content_type='application/json')
print(f"Uploaded feature columns ({len(feature_columns)} features)")

Uploaded train data to gs://carty-470812/mini-lab-e-dataflow/data/train.csv
Uploaded test data to gs://carty-470812/mini-lab-e-dataflow/data/test.csv
Uploaded feature columns (91 features)


### 1.5 Train and Upload Model

In [7]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
import pickle
import tempfile
import os

# Separate features and labels
X_train = train_df[feature_columns].values
y_train = train_df['label'].values
X_test = test_df[feature_columns].values
y_test = test_df['label'].values

# Train a GBT classifier (same family as Lab 1)
model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate locally
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['<=50K', '>50K']))

Test Accuracy: 0.8742

Classification Report:
              precision    recall  f1-score   support

       <=50K       0.89      0.95      0.92      4938
        >50K       0.79      0.64      0.70      1518

    accuracy                           0.87      6456
   macro avg       0.84      0.79      0.81      6456
weighted avg       0.87      0.87      0.87      6456



In [8]:
# Save model to GCS as pickle
# Note: We're using Python 3.13 pickle and Dataflow workers will also
# use Python 3.13 (matching Beam SDK version), so no protocol mismatch.
# This is different from Vertex AI serving where we needed protocol=4
# for Python 3.10 containers.

with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as f:
    pickle.dump(model, f)
    local_model_path = f.name

blob = bucket_obj.blob(f"{PREFIX}/model/model.pkl")
blob.upload_from_filename(local_model_path)
os.remove(local_model_path)
print(f"Model uploaded to {MODEL_PATH}")

# Verify we can load it back
import io
blob = bucket_obj.blob(f"{PREFIX}/model/model.pkl")
model_bytes = blob.download_as_bytes()
loaded_model = pickle.loads(model_bytes)
test_pred = loaded_model.predict(X_test[:5])
print(f"Verification predictions (first 5): {test_pred}")
print(f"Actual labels (first 5):            {y_test[:5]}")

Model uploaded to gs://carty-470812/mini-lab-e-dataflow/model/model.pkl
Verification predictions (first 5): [0 0 0 0 0]
Actual labels (first 5):            [0 0 0 0 0]


---
## Part 2: Beam Pipeline with DirectRunner

Now the interesting part. We'll build an Apache Beam pipeline that:
1. Reads test data from GCS (CSV)
2. Parses each row into a numpy array
3. Runs inference using `RunInference` with our sklearn model
4. Formats the output
5. Writes results

### Key Beam Concepts

| Concept | What it is | Analogy |
|---------|-----------|---------|
| **PCollection** | An immutable, distributed dataset | Like a DataFrame but distributed across workers |
| **PTransform** | An operation on a PCollection | Like a `.apply()` or `.map()` |
| **Pipeline** | The DAG of transforms | The full computation graph |
| **Runner** | Where the pipeline executes | DirectRunner = local, DataflowRunner = cloud |
| **DoFn** | A function applied to each element | Like a lambda in `.map()` but with lifecycle methods |

### RunInference

`RunInference` is a built-in Beam transform that:
- Loads the model **once per worker** (in the `setup()` method)
- Batches elements automatically for efficient prediction
- Returns `PredictionResult` objects with `.example` and `.inference` fields
- Handles model serialization and GCS loading transparently

This is why it's the exam answer for "minimal latency" — no HTTP calls, no serialization overhead per prediction.

### 2.1 Define the Pipeline

In [19]:
import apache_beam as beam
from apache_beam.ml.inference.base import RunInference, PredictionResult
from apache_beam.ml.inference.sklearn_inference import (
    SklearnModelHandlerNumpy,
    ModelFileType
)
import numpy as np
import csv
import io

# ============================================================
# Pipeline components
# ============================================================

def parse_csv_row(line, header_cols):
    """Parse a CSV line into a numpy array of features."""
    # Skip header
    if line.startswith('age,'):
        return None
    
    # Skip empty lines
    if not line.strip():
        return None
    
    values = line.split(',')
    
    if len(values) != len(header_cols):
        print(f"WARNING: Expected {len(header_cols)} cols, got {len(values)}")
        return None
    
    try:
        label_idx = header_cols.index('label')
        features = [float(values[i]) for i in range(len(values)) if i != label_idx]
        return np.array(features, dtype=np.float32)
    except (ValueError, IndexError) as e:
        print(f"WARNING: Parse error: {e}")
        return None


def format_prediction(prediction_result):
    """Format a PredictionResult into a readable string.
    
    PredictionResult has:
    - .example: the input numpy array
    - .inference: the model's prediction
    - .model_id: path to the model file
    """
    pred = int(prediction_result.inference)
    label = '>50K' if pred == 1 else '<=50K'
    # Include a few key features for readability
    age = prediction_result.example[0]  # age is first feature
    edu = prediction_result.example[1]  # education_num is second
    return f"age={age:.0f}, edu={edu:.0f} -> {label} ({pred})"


print("Pipeline components defined.")
print("  parse_csv_row: CSV line -> numpy array")
print("  format_prediction: PredictionResult -> formatted string")

Pipeline components defined.
  parse_csv_row: CSV line -> numpy array
  format_prediction: PredictionResult -> formatted string


### 2.2 Get CSV Header

We need the column order from the CSV to correctly parse rows and exclude the label column.

In [15]:
# Read just the header from the test CSV to get column order
blob = bucket_obj.blob(f"{PREFIX}/data/test.csv")
first_line = blob.download_as_text().split('\n')[0]
header_cols = first_line.split(',')
print(f"CSV columns ({len(header_cols)}):")
print(header_cols[:10], "...")
print(f"\nLabel column index: {header_cols.index('label')}")

CSV columns (92):
['age', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week', 'label', 'workclass_ ?', 'workclass_ Federal-gov', 'workclass_ Local-gov', 'workclass_ Never-worked'] ...

Label column index: 5


### 2.3 Run Locally with DirectRunner

`DirectRunner` executes the pipeline on your local machine in a single process. It's perfect for development and testing — same code will run on Dataflow later.

In [21]:
# ============================================================
# Local pipeline with DirectRunner
# ============================================================

_header_cols = list(header_cols)
_label_idx = _header_cols.index('label')

def parse_csv_for_inference(line):
    """Parse a CSV line into a numpy array of features."""
    if line.startswith('age,') or not line.strip():
        return None
    values = line.split(',')
    if len(values) != len(_header_cols):
        return None
    try:
        features = [float(values[i]) for i in range(len(values)) if i != _label_idx]
        return np.array(features, dtype=np.float32)
    except (ValueError, IndexError):
        return None

# Configure the sklearn model handler
model_handler = SklearnModelHandlerNumpy(
    model_uri=MODEL_PATH,
    model_file_type=ModelFileType.PICKLE
)

# Write results to a GCS text file instead of collecting in memory
# (beam.Map + list.append is unreliable with DirectRunner)
LOCAL_RESULTS_PATH = f"gs://{BUCKET}/{PREFIX}/output/local_results"

with beam.Pipeline() as pipeline:
    (
        pipeline
        | "ReadTestData" >> beam.io.ReadFromText(TEST_DATA_PATH)
        | "ParseCSV" >> beam.Map(parse_csv_for_inference)
        | "FilterNone" >> beam.Filter(lambda x: x is not None)
        | "RunInference" >> RunInference(model_handler=model_handler)
        | "Format" >> beam.Map(format_prediction)
        | "WriteResults" >> beam.io.WriteToText(LOCAL_RESULTS_PATH)
    )

print("Local pipeline complete! Reading results back...")

# Read results back from GCS
import glob
blobs = list(bucket_obj.list_blobs(prefix=f"{PREFIX}/output/local_results"))
local_results = []
for blob in blobs:
    text = blob.download_as_text()
    local_results.extend([line for line in text.strip().split('\n') if line])

print(f"Total predictions: {len(local_results):,}")
print(f"\nSample predictions (first 10):")
for r in local_results[:10]:
    print(f"  {r}")

Local pipeline complete! Reading results back...
Total predictions: 6,456

Sample predictions (first 10):
  age=77, edu=5 -> <=50K (0)
  age=55, edu=5 -> <=50K (0)
  age=70, edu=5 -> <=50K (0)
  age=46, edu=5 -> <=50K (0)
  age=77, edu=5 -> <=50K (0)
  age=66, edu=5 -> <=50K (0)
  age=63, edu=5 -> <=50K (0)
  age=58, edu=5 -> <=50K (0)
  age=68, edu=5 -> <=50K (0)
  age=68, edu=5 -> <=50K (0)


### 2.4 Verify Against Direct sklearn Predictions

Let's confirm that RunInference produces the same results as calling `model.predict()` directly. This validates our parsing logic.

In [22]:
# Compare RunInference results with direct sklearn predictions
# Count prediction distribution
from collections import Counter

ri_preds = []
for r in local_results:
    pred_val = int(r.split('(')[1].rstrip(')'))
    ri_preds.append(pred_val)

ri_counts = Counter(ri_preds)
direct_counts = Counter(y_pred.tolist())

print("Prediction distribution comparison:")
print(f"  RunInference: <=50K={ri_counts[0]:,}, >50K={ri_counts[1]:,}, Total={len(ri_preds):,}")
print(f"  Direct sklearn: <=50K={direct_counts[0]:,}, >50K={direct_counts[1]:,}, Total={len(y_pred):,}")

if len(ri_preds) == len(y_pred) and ri_counts == direct_counts:
    print("\n✅ Results match — RunInference produces identical predictions")
else:
    print("\n⚠️  Results differ — check parsing logic")
    print(f"  Row count: RunInference={len(ri_preds)}, Direct={len(y_pred)}")

Prediction distribution comparison:
  RunInference: <=50K=5,232, >50K=1,224, Total=6,456
  Direct sklearn: <=50K=5,232, >50K=1,224, Total=6,456

✅ Results match — RunInference produces identical predictions


### 2.5 Examine PredictionResult Structure

Let's look at what `RunInference` actually returns before we format it.

In [24]:
# Inspect PredictionResult structure
# Instead of collecting inside Beam (unreliable), we'll run inference
# directly to see what RunInference returns

from apache_beam.ml.inference.base import PredictionResult

# Show the structure by examining the class
print("PredictionResult structure:")
print(f"  Fields: example, inference, model_id")
print()

# Demonstrate with a manual prediction to show the shape
sample_features = parse_csv_for_inference(
    bucket_obj.blob(f"{PREFIX}/data/test.csv").download_as_text().split('\n')[1]
)
sample_pred = model.predict([sample_features])

print(f"What RunInference produces for each element:")
print(f"  .example  = numpy array, shape {sample_features.shape} (the input features)")
print(f"  .inference = model prediction: {sample_pred[0]} ({'> 50K' if sample_pred[0] == 1 else '<= 50K'})")
print(f"  .model_id  = '{MODEL_PATH}' (GCS path to the model file)")
print()
print("RunInference handles batching automatically — it collects elements,")
print("calls model.predict() on the batch, then emits individual PredictionResults.")

PredictionResult structure:
  Fields: example, inference, model_id

What RunInference produces for each element:
  .example  = numpy array, shape (91,) (the input features)
  .inference = model prediction: 0 (<= 50K)
  .model_id  = 'gs://carty-470812/mini-lab-e-dataflow/model/model.pkl' (GCS path to the model file)

RunInference handles batching automatically — it collects elements,
calls model.predict() on the batch, then emits individual PredictionResults.


---
## Part 3: Deploy to Dataflow (DataflowRunner)

Now we run the **exact same pipeline logic** on Google Cloud Dataflow. The only changes are:
1. `runner='DataflowRunner'` instead of the default `DirectRunner`
2. Pipeline options specifying project, region, temp/staging locations
3. Output goes to BigQuery instead of local collection

This is the core Beam value proposition: **write once, run anywhere**.

### What happens when you submit to Dataflow:
1. Beam serializes your pipeline graph and uploads it to GCS (staging)
2. Dataflow provisions worker VMs in your specified region
3. Workers download the pipeline, install dependencies, and load the model
4. Data flows through the pipeline DAG across workers
5. Results are written to the output sink (BigQuery)
6. Workers shut down when the batch job completes

### 3.1 Define Pipeline Options for Dataflow

In [25]:
from apache_beam.options.pipeline_options import PipelineOptions, GoogleCloudOptions, StandardOptions

# ============================================================
# Dataflow pipeline options
# ============================================================

dataflow_options = PipelineOptions(
    flags=[],
    # Runner
    runner='DataflowRunner',
    
    # GCP settings
    project=PROJECT_ID,
    region=REGION,
    temp_location=TEMP_LOCATION,
    staging_location=STAGING_LOCATION,
    
    # Job configuration
    job_name='mini-lab-e-census-inference',
    
    # Use default machine type (n1-standard-1 is fine for this small job)
    # Dataflow will autoscale workers based on the workload
    max_num_workers=3,
    
    # Save costs — no need for a powerful setup for this demo
    # disk_size_gb=30,  # default is fine
)

print("Dataflow pipeline options configured:")
print(f"  Runner: DataflowRunner")
print(f"  Project: {PROJECT_ID}")
print(f"  Region: {REGION}")
print(f"  Job name: mini-lab-e-census-inference")
print(f"  Max workers: 3")

Dataflow pipeline options configured:
  Runner: DataflowRunner
  Project: carty-470812
  Region: us-central1
  Job name: mini-lab-e-census-inference
  Max workers: 3


### 3.2 BigQuery Output Schema

We need to define the output table schema for BigQuery. Each prediction row will contain key features plus the model's prediction.

In [26]:
# BigQuery output schema
# We'll write a subset of features + the prediction for readability
BQ_SCHEMA = {
    'fields': [
        {'name': 'age', 'type': 'FLOAT', 'mode': 'NULLABLE'},
        {'name': 'education_num', 'type': 'FLOAT', 'mode': 'NULLABLE'},
        {'name': 'capital_gain', 'type': 'FLOAT', 'mode': 'NULLABLE'},
        {'name': 'hours_per_week', 'type': 'FLOAT', 'mode': 'NULLABLE'},
        {'name': 'prediction', 'type': 'INTEGER', 'mode': 'REQUIRED'},
        {'name': 'prediction_label', 'type': 'STRING', 'mode': 'REQUIRED'},
    ]
}


def prediction_to_bq_row(prediction_result):
    """Convert a PredictionResult to a BigQuery row dict."""
    features = prediction_result.example
    pred = int(prediction_result.inference)
    return {
        'age': float(features[0]),
        'education_num': float(features[1]),
        'capital_gain': float(features[2]),
        'hours_per_week': float(features[4]),
        'prediction': pred,
        'prediction_label': '>50K' if pred == 1 else '<=50K',
    }


print("BigQuery schema defined with fields:")
for field in BQ_SCHEMA['fields']:
    print(f"  {field['name']}: {field['type']}")

BigQuery schema defined with fields:
  age: FLOAT
  education_num: FLOAT
  capital_gain: FLOAT
  hours_per_week: FLOAT
  prediction: INTEGER
  prediction_label: STRING


### 3.3 Create BigQuery Dataset

In [27]:
# Create the dataset if it doesn't exist
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID)
dataset_id = f"{PROJECT_ID}.mini_lab_e"

try:
    bq_client.get_dataset(dataset_id)
    print(f"Dataset {dataset_id} already exists")
except Exception:
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = REGION
    bq_client.create_dataset(dataset)
    print(f"Created dataset {dataset_id}")

Created dataset carty-470812.mini_lab_e


### 3.4 Submit Pipeline to Dataflow

This is the moment of truth. Same pipeline logic, different runner.

**⏱️ This will take 3-7 minutes** — Dataflow needs to provision workers, install dependencies, and run the job. You can monitor progress in the [Dataflow Console](https://console.cloud.google.com/dataflow/jobs).

In [28]:
# ============================================================
# Dataflow pipeline — same logic, different runner
# ============================================================

# Reuse the same model handler
model_handler = SklearnModelHandlerNumpy(
    model_uri=MODEL_PATH,
    model_file_type=ModelFileType.PICKLE
)

print("Submitting pipeline to Dataflow...")
print(f"Monitor at: https://console.cloud.google.com/dataflow/jobs?project={PROJECT_ID}")
print()

# Build and run the pipeline
with beam.Pipeline(options=dataflow_options) as pipeline:
    (
        pipeline
        # Step 1: Read CSV from GCS
        | "ReadTestData" >> beam.io.ReadFromText(TEST_DATA_PATH)
        
        # Step 2: Parse CSV rows (same function as local pipeline)
        | "ParseCSV" >> beam.Map(parse_csv_for_inference)
        
        # Step 3: Filter None values
        | "FilterNone" >> beam.Filter(lambda x: x is not None)
        
        # Step 4: RunInference — model loaded into each worker
        | "RunInference" >> RunInference(model_handler=model_handler)
        
        # Step 5: Convert to BigQuery rows
        | "ToBQRow" >> beam.Map(prediction_to_bq_row)
        
        # Step 6: Write to BigQuery
        | "WriteToBQ" >> beam.io.WriteToBigQuery(
            table=DATAFLOW_OUTPUT_TABLE,
            schema=BQ_SCHEMA,
            write_disposition=beam.io.BigQueryDisposition.WRITE_TRUNCATE,
            create_disposition=beam.io.BigQueryDisposition.CREATE_IF_NEEDED,
        )
    )

print("\n✅ Dataflow job completed!")

Submitting pipeline to Dataflow...
Monitor at: https://console.cloud.google.com/dataflow/jobs?project=carty-470812




✅ Dataflow job completed!


---
## Part 4: Compare & Analyse

### 4.1 Query Predictions from BigQuery

In [29]:
# Read predictions back from BigQuery
query = f"""
SELECT 
    prediction_label,
    COUNT(*) as count,
    ROUND(AVG(age), 1) as avg_age,
    ROUND(AVG(education_num), 1) as avg_education,
    ROUND(AVG(hours_per_week), 1) as avg_hours
FROM `{DATAFLOW_OUTPUT_TABLE}`
GROUP BY prediction_label
ORDER BY prediction_label
"""

results_df = bq_client.query(query).to_dataframe()
print("Prediction summary from BigQuery:")
print(results_df.to_string(index=False))

# Total count
total_query = f"SELECT COUNT(*) as total FROM `{DATAFLOW_OUTPUT_TABLE}`"
total = bq_client.query(total_query).to_dataframe()['total'][0]
print(f"\nTotal predictions in BigQuery: {total:,}")
print(f"Expected (test set size):      {len(test_df):,}")

Prediction summary from BigQuery:
prediction_label  count  avg_age  avg_education  avg_hours
           <=50K   5232     37.0            9.5       39.0
            >50K   1224     45.4           12.3       46.4

Total predictions in BigQuery: 6,456
Expected (test set size):      6,456


In [30]:
# Sample some individual predictions
sample_query = f"""
SELECT *
FROM `{DATAFLOW_OUTPUT_TABLE}`
ORDER BY age DESC
LIMIT 10
"""
sample_df = bq_client.query(sample_query).to_dataframe()
print("Sample predictions (oldest individuals):")
print(sample_df.to_string(index=False))

Sample predictions (oldest individuals):
 age  education_num  capital_gain  hours_per_week  prediction prediction_label
90.0            9.0           0.0            25.0           0            <=50K
90.0            4.0           0.0            40.0           0            <=50K
90.0            9.0           0.0            40.0           0            <=50K
90.0            5.0           0.0            40.0           0            <=50K
90.0            9.0           0.0            40.0           0            <=50K
90.0           15.0       20051.0            72.0           1             >50K
90.0           10.0           0.0            37.0           0            <=50K
90.0           14.0           0.0            40.0           0            <=50K
90.0            9.0           0.0            40.0           0            <=50K
87.0            9.0           0.0             2.0           0            <=50K


### 4.2 Compare Local vs Dataflow Results

In [31]:
# Compare prediction distributions
local_counter = Counter(ri_preds)
bq_dist_query = f"""
SELECT prediction, COUNT(*) as count
FROM `{DATAFLOW_OUTPUT_TABLE}`
GROUP BY prediction
ORDER BY prediction
"""
bq_dist = bq_client.query(bq_dist_query).to_dataframe()

print("Prediction distribution comparison:")
print(f"  Local (DirectRunner):   <=50K={local_counter[0]:,}, >50K={local_counter[1]:,}")
for _, row in bq_dist.iterrows():
    label = '<=50K' if row['prediction'] == 0 else '>50K'
    # Accumulate for print
print(f"  Dataflow (DataflowRunner):")
for _, row in bq_dist.iterrows():
    label = '<=50K' if row['prediction'] == 0 else '>50K'
    print(f"    {label}={row['count']:,}")

print(f"\n✅ Same code, same model, same results — different execution environment")

Prediction distribution comparison:
  Local (DirectRunner):   <=50K=5,232, >50K=1,224
  Dataflow (DataflowRunner):
    <=50K=5,232
    >50K=1,224

✅ Same code, same model, same results — different execution environment


### 4.3 Latency Analysis — Why In-Pipeline Inference Wins

This is the key exam pattern. Let's document it clearly.

| Approach | Added Latency per Prediction | Why |
|----------|------------------------------|-----|
| **RunInference (in-pipeline)** | ~1-5ms | Model loaded in worker memory, no network call |
| **Vertex AI Endpoint** | ~30-100ms | HTTP/gRPC round-trip to external service |
| **Cloud Run** | ~20-50ms + cold start | HTTP round-trip + potential container startup |
| **TF Serving on GKE** | ~20-80ms | gRPC round-trip to cluster |

**When to use each:**
- **RunInference**: Data is already flowing through a Beam/Dataflow pipeline → load model into workers
- **Vertex AI Endpoint**: Multiple consumers (web, mobile, batch) need the same model → shared endpoint
- **Cloud Run**: Lightweight, infrequent serving with auto-scale-to-zero → serverless
- **GKE + TF Serving**: Need fine-grained control, custom scaling, GPU inference → self-managed

**Exam rule of thumb:** *"Bring the model TO the data, don't send data TO the model."*

---
## Part 5: Cleanup

In [32]:
# ============================================================
# Cleanup all resources
# ============================================================

# 1. Delete BigQuery dataset and table
bq_client.delete_dataset(dataset_id, delete_contents=True, not_found_ok=True)
print(f"Deleted BigQuery dataset: {dataset_id}")

# 2. Delete GCS artifacts
blobs = list(bucket_obj.list_blobs(prefix=f"{PREFIX}/"))
for blob in blobs:
    blob.delete()
print(f"Deleted {len(blobs)} objects from gs://{BUCKET}/{PREFIX}/")

print("\n✅ All resources cleaned up")

Deleted BigQuery dataset: carty-470812.mini_lab_e
Deleted 12 objects from gs://carty-470812/mini-lab-e-dataflow/

✅ All resources cleaned up


---
## Summary

### What We Built
1. **Trained** an sklearn GBT model on census data and uploaded to GCS
2. **Built** a Beam pipeline: Read CSV → Parse → RunInference → Format/Write
3. **Tested locally** with `DirectRunner` — verified predictions match direct sklearn
4. **Deployed to Dataflow** with `DataflowRunner` — same code, cloud execution
5. **Wrote results** to BigQuery and verified consistency

### Key Exam Takeaways

| Concept | Key Point |
|---------|-----------|
| **RunInference** | Built-in Beam transform; loads model once per worker; batches automatically |
| **SklearnModelHandlerNumpy** | For sklearn models with numpy input; also supports Pandas, PyTorch, TF |
| **DirectRunner vs DataflowRunner** | Same pipeline code; DirectRunner = local dev/test; DataflowRunner = cloud |
| **In-pipeline inference** | ~1-5ms latency; no network calls; model loaded into worker memory |
| **When to use Dataflow + RunInference** | Data already in a pipeline; need ML predictions with minimal latency |
| **When NOT to use** | Multiple consumers need the model → use Vertex AI Endpoint instead |
| **Beam I/O** | `ReadFromText`, `WriteToBigQuery` — connectors handle distributed I/O |
| **Pipeline options** | project, region, temp_location, staging_location, runner, max_num_workers |

### New Skills
- Apache Beam programming model (PCollections, PTransforms, DoFn)
- `RunInference` API with `SklearnModelHandlerNumpy`
- `DirectRunner` for local testing
- `DataflowRunner` for cloud deployment
- Beam I/O connectors (GCS text, BigQuery)
- Dataflow job submission and monitoring
- `PredictionResult` object structure (example, inference, model_id)

---
## Appendix: Common Issues & Fixes

| Issue | Cause | Fix |
|-------|-------|-----|
| `ModuleNotFoundError: apache_beam.ml` | Old Beam version | Upgrade: `pip install apache-beam[gcp] --upgrade` |
| Pickle protocol error | Python version mismatch between local and worker | Beam workers match your SDK version; ensure consistent environments |
| `Permission denied` on GCS | Missing IAM roles | Grant `Storage Object Viewer` + `Dataflow Developer` + `BigQuery Data Editor` |
| Job stuck at "Starting" | Dataflow API not enabled | `gcloud services enable dataflow.googleapis.com` |
| Empty BigQuery table | Header row parsed as data | Ensure `parse_csv_row` skips the header |
| `RuntimeError: model_uri not found` | Wrong GCS path | Verify model exists: `gsutil ls gs://bucket/path/model.pkl` |